# Comparação de AutoML: FLAML e Auto-Sklearn

**Dataset:** Iris  
**Objetivo:** comparar a acurácia, o tempo de execução e os modelos encontrados pelas duas ferramentas.

## 1. Preparando os dados

In [1]:
from pathlib import Path
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

pasta = Path('/content') if Path('/content').exists() else Path.cwd()
iris = load_iris()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    iris.data, iris.target, test_size=0.2,
    stratify=iris.target, random_state=42
)

# Os dados serão iguais para as duas ferramentas.
np.savez(pasta / 'iris_split.npz', X_train=X_treino, X_test=X_teste,
         y_train=y_treino, y_test=y_teste)

print('Treino:', len(y_treino), 'amostras')
print('Teste:', len(y_teste), 'amostras')

Treino: 120 amostras
Teste: 30 amostras


## 2. Instalando as bibliotecas

O Auto-Sklearn usa uma versão mais antiga do Python. Para não misturar as dependências, vou usar um ambiente para cada ferramenta. A instalação é feita só na primeira vez.

In [2]:
import subprocess

micromamba = pasta / 'bin' / 'micromamba'
if not micromamba.exists():
    subprocess.run(
        f'curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest '
        f'| tar -xj -C {pasta} bin/micromamba',
        shell=True, check=True, executable='/bin/bash'
    )

ambiente_flaml = pasta / 'env_flaml'
if not (ambiente_flaml / 'bin/python').exists():
    subprocess.run([
        str(micromamba), 'create', '-y', '-p', str(ambiente_flaml),
        '-c', 'conda-forge', 'python=3.11', 'pip', 'numpy=1.26.4',
        'scipy=1.13.1', 'scikit-learn=1.4.2'
    ], check=True)

# A opção [automl] é necessária para importar AutoML.
subprocess.run([
    str(ambiente_flaml / 'bin/python'), '-m', 'pip', 'install', '-q',
    'flaml[automl]==2.3.6', 'lightgbm==4.6.0'
], check=True)
subprocess.run([
    str(ambiente_flaml / 'bin/python'), '-c',
    'from flaml import AutoML; print("FLAML instalado")'
], check=True)

ambiente_auto = pasta / 'env_autosklearn'
if not (ambiente_auto / 'bin/python').exists():
    subprocess.run([
        str(micromamba), 'create', '-y', '-p', str(ambiente_auto),
        '-c', 'conda-forge', 'python=3.9', 'auto-sklearn=0.15.0'
    ], check=True)

print('Auto-Sklearn instalado')

Auto-Sklearn instalado


## 3. Treinando e avaliando

Cada ferramenta terá **60 segundos de busca**. O código abaixo executa os treinamentos com a mesma divisão do Iris e calcula acurácia e relatório de classificação.

In [3]:
# Pequeno código usado pelos dois ambientes Python.
# Isso permite fazer o mesmo teste sem misturar as bibliotecas.
codigo_teste = r'''
import json
import sys
import time
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

pasta = Path('/content') if Path('/content/iris_split.npz').exists() else Path(__file__).parent
ferramenta = sys.argv[1]
dados = np.load(pasta / 'iris_split.npz')
X_treino, X_teste = dados['X_train'], dados['X_test']
y_treino, y_teste = dados['y_train'], dados['y_test']

if ferramenta == 'flaml':
    from flaml import AutoML
    from flaml.automl.data import get_output_from_log
    modelo = AutoML()
    inicio = time.perf_counter()
    modelo.fit(X_train=X_treino, y_train=y_treino,
               task='classification', metric='accuracy',
               time_budget=60, n_jobs=1, seed=42,
               estimator_list=['lgbm', 'rf', 'extra_tree'],
               eval_method='holdout', split_ratio=0.25,
               log_file_name=str(pasta / 'busca_flaml.log'),
               log_type='all', verbose=0)
    tempo = time.perf_counter() - inicio
    previsao = modelo.predict(X_teste)
    try:
        tempos, _, _, _, _ = get_output_from_log(
            str(pasta / 'busca_flaml.log'), time_budget=360)
        tentativas = len(tempos)
    except Exception:
        tentativas = None
    nome_modelo = modelo.best_estimator
    detalhes = str(modelo.best_config)
    quantidade_modelos = 1

elif ferramenta == 'autosklearn':
    import autosklearn.classification
    from autosklearn.metrics import accuracy
    modelo = autosklearn.classification.AutoSklearnClassifier(
        time_left_for_this_task=60, per_run_time_limit=15,
        memory_limit=2048, metric=accuracy, seed=42,
        n_jobs=1, initial_configurations_via_metalearning=0,
        resampling_strategy='holdout',
        resampling_strategy_arguments={'train_size': 0.75},
        ensemble_size=20
    )
    inicio = time.perf_counter()
    modelo.fit(X_treino, y_treino, dataset_name='iris')
    tempo = time.perf_counter() - inicio
    previsao = modelo.predict(X_teste)
    ensemble = modelo.get_models_with_weights()
    tentativas = len(modelo.cv_results_['mean_test_score'])
    nome_modelo = 'Ensemble do Auto-Sklearn'
    detalhes = '\n'.join(
        f'Peso {peso:.2f}: {str(m)[:700]}' for peso, m in ensemble
    )
    quantidade_modelos = len(ensemble)
else:
    raise ValueError('Ferramenta desconhecida')

resultado = {
    'acuracia': float(accuracy_score(y_teste, previsao)),
    'tempo_treino': round(tempo, 2),
    'tentativas': tentativas,
    'modelo': nome_modelo,
    'detalhes': detalhes,
    'quantidade_modelos': quantidade_modelos,
    'relatorio': classification_report(
        y_teste, previsao,
        target_names=['setosa', 'versicolor', 'virginica'],
        zero_division=0
    )
}
(pasta / f'resultado_{ferramenta}.json').write_text(
    json.dumps(resultado, ensure_ascii=False), encoding='utf-8'
)
'''
(pasta / 'teste_automl.py').write_text(codigo_teste, encoding='utf-8')
print('Código dos testes preparado.')

Código dos testes preparado.


In [4]:
import json
import time
import psutil
import pandas as pd
from IPython.display import display

# Executa o teste e mede uma estimativa do pico de RAM.
def testar(ferramenta, ambiente):
    arquivo = pasta / f'resultado_{ferramenta}.json'
    arquivo.unlink(missing_ok=True)
    comando = [str(ambiente / 'bin/python'), str(pasta / 'teste_automl.py'), ferramenta]

    inicio = time.perf_counter()
    pico = 0
    with (pasta / f'erro_{ferramenta}.txt').open('w') as log:
        processo = subprocess.Popen(comando, stdout=log, stderr=subprocess.STDOUT)
        while processo.poll() is None:
            try:
                proc = psutil.Process(processo.pid)
                ram = sum(p.memory_info().rss for p in [proc] + proc.children(recursive=True))
                pico = max(pico, ram)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
            time.sleep(0.2)

    if processo.returncode != 0 or not arquivo.exists():
        print((pasta / f'erro_{ferramenta}.txt').read_text(errors='replace')[-3000:])
        raise RuntimeError(f'Erro no teste do {ferramenta}')

    resultado = json.loads(arquivo.read_text(encoding='utf-8'))
    resultado['tempo_total'] = round(time.perf_counter() - inicio, 2)
    resultado['memoria'] = round(pico / 1024**2, 1)
    return resultado

flaml = testar('flaml', ambiente_flaml)
autosklearn = testar('autosklearn', ambiente_auto)

resumo = pd.DataFrame([
    {'Ferramenta': 'FLAML', 'Acurácia': f"{flaml['acuracia']:.2%}",
     'Treino (s)': flaml['tempo_treino'], 'Total (s)': flaml['tempo_total'],
     'RAM aprox. (MB)': flaml['memoria'], 'Testes realizados': flaml['tentativas']},
    {'Ferramenta': 'Auto-Sklearn', 'Acurácia': f"{autosklearn['acuracia']:.2%}",
     'Treino (s)': autosklearn['tempo_treino'], 'Total (s)': autosklearn['tempo_total'],
     'RAM aprox. (MB)': autosklearn['memoria'], 'Testes realizados': autosklearn['tentativas']}
])
display(resumo)

for nome, resultado in [('FLAML', flaml), ('Auto-Sklearn', autosklearn)]:
    print('\n' + nome)
    print(resultado['relatorio'])
    print('Modelo:', resultado['modelo'])
    print('Detalhes:', resultado['detalhes'])
    print('Modelos no resultado:', resultado['quantidade_modelos'])

if flaml['acuracia'] == autosklearn['acuracia']:
    print('\nResultado: empate na acurácia.')
elif flaml['acuracia'] > autosklearn['acuracia']:
    print('\nResultado: FLAML teve maior acurácia.')
else:
    print('\nResultado: Auto-Sklearn teve maior acurácia.')

,Ferramenta,Acurácia,Treino (s),Total (s),RAM aprox. (MB),Testes realizados
0,FLAML,90.00%,60.05,62.16,178.1,2438
1,Auto-Sklearn,96.67%,55.50,57.71,479.3,15



FLAML
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Modelo: rf
Detalhes: {'n_estimators': 27, 'max_leaves': 32, 'max_features': np.float64(0.267847370832647), 'criterion': np.str_('entropy')}
Modelos no resultado: 1

Auto-Sklearn
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Modelo: Ensemble do Auto-Sklearn
Detalhes: Peso 0.20:

## 4. Conclusão

Para realizar a comparação, utilizei o conjunto de dados Iris, com 120 amostras para treinamento e 30 para teste. O FLAML obteve uma acurácia de **90%**, enquanto o Auto-Sklearn alcançou **96,67%**. Portanto, não houve empate, e o Auto-Sklearn apresentou o melhor desempenho, acertando 29 das 30 amostras do teste, enquanto o FLAML acertou 27.

Em relação ao tempo de execução, o FLAML levou **62,16 segundos** no total, enquanto o Auto-Sklearn levou **57,71 segundos**, sendo um pouco mais rápido. Porém, o FLAML utilizou aproximadamente **178,1 MB de memória RAM**, enquanto o Auto-Sklearn utilizou **479,3 MB**. Além disso, o FLAML realizou 2.438 tentativas de busca, contra 15 do Auto-Sklearn. Isso mostra que o FLAML fez uma busca mais extensa e consumiu menos memória, mesmo tendo apresentado uma acurácia menor.

Na parte de análise dos modelos, achei o FLAML mais fácil de entender, pois ele mostrou diretamente o modelo escolhido, que foi o Random Forest, junto com seus parâmetros. Já o Auto-Sklearn apresentou um ensemble formado por sete modelos, tornando a análise um pouco mais complicada. Com isso, concluí que o Auto-Sklearn foi melhor em acurácia e tempo de execução neste teste, enquanto o FLAML se destacou pelo menor consumo de memória e pela facilidade de visualizar o modelo escolhido.